# ViGoEmotions — Kaggle 28 nhãn, neural 20 epoch

Notebook độc lập chạy **ba mô hình cho 28 nhãn**: TF-IDF + OvR Logistic Regression, BiGRU và PhoBERT. Hai neural đều chạy đủ **20 epoch**; TF-IDF giữ solver liblinear/max_iter=1000, không có epoch neural tương đương.

Cả notebook 27 và 28 nhãn dùng duy nhất `configs/kaggle_equal_20ep.json`, cùng chọn checkpoint bằng validation Macro-F1 trên 27 cảm xúc, ngưỡng 0,5, hòa chọn epoch sớm. Notebook này có workspace, manifest, kết quả và ZIP riêng; không phải chờ hoặc chạy notebook còn lại.

Giữ Dataset/notebook Private. Attach `vigoemotions_kaggle_equal_20ep.zip` cùng ba CSV `id,text,labels`; bật GPU + Internet. Chạy tuần tự và dừng khi lỗi. Không chấm test, không dùng số baseline cũ. Chưa chạy GPU thì chưa có kết quả mới. Xem `docs/kaggle_equal_epochs.md`.

## 1. Nạp code vào workspace mới
Tạo ZIP trên máy bằng `python scripts/package_kaggle_equal_epochs.py`. Attach đúng một Dataset code và một Dataset dữ liệu. Nếu workspace đã tồn tại, không giải nén đè; dùng cell nối workspace sau restart kernel.

In [ ]:
from pathlib import Path
import os, shutil, zipfile

INPUT = Path('/kaggle/input')
ROOT = Path('/kaggle/working/ViGoEmotions28Equal20')
if ROOT.exists():
    raise FileExistsError(f'{ROOT} đã tồn tại. Không giải nén đè lên kết quả; chạy tiếp cell cần thiết hoặc dùng session mới.')

# Kaggle có thể giữ ZIP hoặc tự giải nén ZIP khi tạo Dataset.
archives = list(INPUT.rglob('vigoemotions_kaggle_equal_20ep.zip'))
code_roots = [p.parent.parent for p in INPUT.rglob('scripts/kaggle_equal_epochs.py')]
if len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as archive:
        for item in archive.infolist():
            if not (ROOT / item.filename).resolve().is_relative_to(ROOT.resolve()):
                raise ValueError('Unsafe ZIP path')
        ROOT.mkdir(parents=True)
        archive.extractall(ROOT)
elif not archives and len(code_roots) == 1:
    shutil.copytree(code_roots[0], ROOT)
else:
    raise RuntimeError('Hãy attach đúng một Dataset code chứa vigoemotions_kaggle_equal_20ep.zip hoặc project đã giải nén.')
os.chdir(ROOT)
print('Project:', ROOT)

In [ ]:
from pathlib import Path
import os, sys, subprocess
ROOT = Path('/kaggle/working/ViGoEmotions28Equal20')
if not (ROOT/'scripts/kaggle_equal_epochs.py').is_file():
    raise FileNotFoundError('Nạp project trước; sau restart kernel chỉ chạy cell nối workspace này.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'scripts'))

## 2. Dependencies và GPU/Java
Internet phục vụ cài dependency và tải pretrained PhoBERT/VnCoreNLP. Một GPU được dùng cho mỗi lượt, các model chạy tuần tự. Không đổi loại GPU/package giữa study đã init.

In [ ]:
%pip install -q -r requirements-colab.txt
import shutil, subprocess
if shutil.which('java') is None:
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'default-jre-headless'], check=True)

In [ ]:
import torch, platform
assert torch.cuda.is_available(), 'Bật GPU Kaggle trước khi train neural.'
print('Python:', platform.python_version(), '| GPU:', torch.cuda.get_device_name(0))
subprocess.run(['java', '-version'], check=True)

## 3. Nạp CSV riêng tư
Schema `id,text,labels`; bỏ qua CSV chỉ chứa ID trong gói code. Không hiển thị bình luận nguồn.

In [ ]:
from pathlib import Path
import csv, shutil

sources = {}
for name in ('train.csv', 'val.csv', 'test.csv'):
    candidates = []
    for path in Path('/kaggle/input').rglob(name):
        with path.open(encoding='utf-8-sig', newline='') as handle:
            header = next(csv.reader(handle), None)
        # ZIP code có splits/*.csv chỉ chứa ID, không phải dữ liệu nguồn.
        if path.parent.name == 'splits' and header == ['id']:
            continue
        if header != ['id', 'text', 'labels']:
            raise ValueError(f'{path}: schema phải là id,text,labels')
        candidates.append(path)
    if len(candidates) != 1:
        raise RuntimeError(f'Cần đúng một {name} dữ liệu (id,text,labels) trong Input; tìm thấy {len(candidates)}. Kiểm tra Dataset đã attach.')
    sources[name] = candidates[0]
(ROOT / 'data').mkdir(exist_ok=True)
for name, source in sources.items():
    shutil.copy2(source, ROOT / 'data' / name)
    print('Đã nạp:', name)

DATA_DIR = ROOT/'data'

## 4. Init và đóng băng nguồn cấu hình
`OUT` là namespace mới riêng tư. Có thể đổi tên study con trước init; không đổi hyperparameters giữa study. Config/data/source/runtime khác sẽ dừng thay vì reuse nhầm kết quả cũ.

In [ ]:
import json
import pandas as pd
from IPython.display import display

LABEL_COUNT = 28
OUT = ROOT/'reports/neutral_28_labels/kaggle_equal_20ep/labels28_study'
def run(action, *arguments):
    subprocess.run([sys.executable, '-u', str(ROOT/'scripts/kaggle_equal_epochs.py'), action,
                    '--output', str(OUT), '--data-dir', str(DATA_DIR),
                    '--labels', str(LABEL_COUNT), *map(str, arguments)],
                   cwd=ROOT, check=True)

run('init')
frozen = json.loads((OUT/'config27.json').read_text())
assert frozen['gru']['epochs'] == frozen['phobert']['epochs'] == 20
display(pd.DataFrame([dict(model=name, epochs=frozen[name]['epochs'],
                          batch_size=frozen[name]['batch_size']) for name in ('gru','phobert')]))
print('TF-IDF/LR:', frozen['logistic_regression'])
print('Checkpoint selection:', frozen['checkpoint_selection'])
print('Manifest:', json.loads((OUT/'manifest.json').read_text()))

## 5. Kiểm thử offline giao thức
Fixture kiểm tra profile, tiêu chí chọn checkpoint, chống lẫn nguồn và output TF-IDF riêng. Không phải kết quả mô hình trên dataset thật.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
                '-p', 'test_kaggle_equal_epochs.py', '-v'], cwd=ROOT, check=True)

## Train 28 nhãn — TF-IDF + OvR Logistic Regression
Model hoàn tất được kiểm chứng và giữ nguyên khi chạy lại cell; model dở dang không resume giữa epoch, dùng study mới để chạy lại.

In [ ]:
run('train', '--model', 'tfidf_lr')

## Train 28 nhãn — BiGRU — 20 epoch
Model hoàn tất được kiểm chứng và giữ nguyên khi chạy lại cell; model dở dang không resume giữa epoch, dùng study mới để chạy lại.

In [ ]:
run('train', '--model', 'gru')

## Train 28 nhãn — PhoBERT — 20 epoch
Model hoàn tất được kiểm chứng và giữ nguyên khi chạy lại cell; model dở dang không resume giữa epoch, dùng study mới để chạy lại.

In [ ]:
run('train', '--model', 'phobert')

## 6. Kết quả ba mô hình 28 nhãn
Bảng chỉ dùng TF-IDF/BiGRU/PhoBERT của notebook này, không cần kết quả notebook còn lại để hoàn tất hoặc tải ZIP. Khi đối chiếu 27/28 sau này, dùng emotion27_* và kiểm tra profile, source code, CSV hash, fixed IDs; công bố khác biệt GPU/package nếu hai session khác nhau. Không nhập điểm lịch sử vào bảng.

In [ ]:
run('report')
status = json.loads((OUT/'status.json').read_text())
assert status['status'] == 'completed' and status['expected_runs'] == 3, 'Hoàn tất đủ ba model 28 nhãn trước khi archive.'
display(pd.read_csv(OUT/'comparison.csv'))
print(status)

## 7. Tải toàn bộ bằng chứng/checkpoint riêng tư
ZIP gồm manifest, config, log neural, history, probabilities, checkpoint và vocabulary; không kèm CSV/bình luận nguồn. Giữ Private vì có vocabulary/checkpoint. Tải trước khi mất runtime; không dùng ZIP này làm gói công khai. Không có cell chấm test.

Sau khi tải về, dùng ZIP này để nhập/kiểm chứng và cập nhật report; không sửa tay số điểm cũ thành điểm mới.

In [ ]:
run('archive')
archive = ROOT/'vigoemotions_kaggle_equal20_28labels_private_results.zip'
assert archive.is_file()
print('Tải từ Kaggle Output:', archive, '| MiB:', round(archive.stat().st_size/1024**2, 2))